In [ ]:
# First round crossing
import os
import pandas as pd
import re
import unidecode

# Function to clean, transliterate, and tokenize names into word chains (removes hyphens as well)
def tokenize_name(name):
    if pd.isna(name):
        return []
    
    # Transliterate the name to Latin characters (this helps with encoded characters)
    name = unidecode.unidecode(name)
    
    # Convert to lowercase, remove unnecessary special characters (including hyphens)
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)  # Remove special characters including hyphens
    name = re.sub(r'\s+', ' ', name).strip()  # Replace multiple spaces with a single space
    return name.split()  # Split name into individual word chains

# Function to clean the 'Speaker' column by removing unwanted symbols and hyphens
def clean_speaker_column(speaker):
    if pd.isna(speaker):
        return speaker
    # Attempt to decode/transliterate first
    speaker = unidecode.unidecode(speaker)
    
    # Now remove unwanted symbols and hyphens
    speaker = re.sub(r'[^\w\s]', '', speaker)  # Remove special characters including hyphens
    return speaker

# Function to count matching word chains between source and target rows
def count_matching_chains(source_name, target_name):
    source_chains = tokenize_name(source_name)
    target_chains = tokenize_name(target_name)
    
    # Count how many word chains from the source match word chains in the target
    matching_chains = len(set(source_chains) & set(target_chains))
    
    return matching_chains

# Custom function to handle date parsing
def custom_date_parser(date_str):
    try:
        if "/" in date_str:  # Handle date ranges
            start_date_str, end_date_str = date_str.split("/")
            start_date = pd.to_datetime(start_date_str.strip(), errors='coerce', dayfirst=True)
            end_date = pd.to_datetime(end_date_str.strip(), errors='coerce', dayfirst=True)
            return start_date, end_date
        else:
            parsed_date = pd.to_datetime(date_str.strip(), errors='coerce', dayfirst=True)
            return parsed_date
    except Exception as e:
        print(f"Error parsing date: {date_str} - {e}")
        return pd.NaT

# Paths to the source and target CSV files
source_path = 
target_path = 

# Load the source and target CSVs into DataFrames
source_df = pd.read_csv(source_path)
target_df = pd.read_csv(target_path)

# Parse dates with the custom date parser
source_df['date'] = source_df['date'].apply(lambda x: custom_date_parser(x))
target_df['date'] = target_df['date'].apply(lambda x: pd.to_datetime(x, errors='coerce', dayfirst=True))

# Clean the 'Speaker' column in target_df (remove symbols and hyphens)
target_df['Speaker'] = target_df['Speaker'].apply(lambda x: clean_speaker_column(x))

# Check if "Other institution" column exists before processing it
other_institution_exists = 'Other institution' in target_df.columns

# Initialize counters for matches found and not found
matches_found = 0
matches_not_found = 0

# Iterate through the target DataFrame to find matching chains
for target_index, target_row in target_df.iterrows():
    # Skip rows with 'President' in the 'Speaker' column or with non-null 'Other institution' if the column exists
    if target_row['Speaker'] == 'President' or (other_institution_exists and pd.notna(target_row['Other institution'])):
        continue

    max_matches = 0
    best_match_source_row = None

    for source_index, source_row in source_df.iterrows():
        matching_chains = count_matching_chains(source_row['Name'], target_row['Speaker'])
        if matching_chains > max_matches:
            max_matches = matching_chains
            best_match_source_row = source_row

    # Print the result for this target row
    if best_match_source_row is not None:
        print(f"Target Speaker '{target_row['Speaker']}' matches {max_matches} chains with source name '{best_match_source_row['Name']}'")
    else:
        print(f"No match found for Target Speaker '{target_row['Speaker']}'")

    # If a match is found with a significant number of chains, fill in missing data
    if max_matches > 0 and best_match_source_row is not None:
        if pd.isna(target_row['Country']):
            target_df.at[target_index, 'Country'] = best_match_source_row['Country']
        if pd.isna(target_row['Party']):
            target_df.at[target_index, 'Party'] = best_match_source_row['Party']
        matches_found += 1
    else:
        matches_not_found += 1

# Create output directory if it does not exist
output_dir = r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\preprocessing\2009-2014_CRE-7\2014\2014_crossed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the updated target DataFrame back to a new CSV file in the new directory
output_path = os.path.join(output_dir, "2014_crossed.csv")
target_df.to_csv(output_path, index=False)

print(f"\nUpdated data saved to {output_path}")

# Print the number of matches found and not found
print(f"Total matches found: {matches_found}")
print(f"Total matches not found: {matches_not_found}")

# Print a sample of the updated DataFrame
print("\nUpdated DataFrame (sample):")
print(target_df.head())




In [ ]:
#Second round crossing
import os
import pandas as pd
import re
import unidecode

# Function to try reading the CSV file with different encodings
def read_csv_with_multiple_encodings(file_path):
    encodings = ['utf-8', 'ISO-8859-1', 'windows-1252', 'latin1']
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding), encoding
        except UnicodeDecodeError as e:
            print(f"Failed to read {file_path} with encoding {encoding}: {e}")
    raise UnicodeDecodeError(f"Could not decode {file_path} with the tried encodings.")

# Function to clean and tokenize names into word chains (removes hyphens and special characters)
def tokenize_name(name):
    if pd.isna(name):
        return []
    
    # Transliterate the name to Latin characters
    name = unidecode.unidecode(name)
    
    # Convert to lowercase, remove unnecessary special characters (including hyphens)
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)  # Remove special characters including hyphens
    name = re.sub(r'\s+', ' ', name).strip()  # Replace multiple spaces with a single space and strip leading/trailing spaces
    return name.split()  # Split name into individual word chains

# Function to clean the 'Speaker' column by removing unwanted symbols and hyphens
def clean_speaker_column(speaker):
    if pd.isna(speaker):
        return speaker
    speaker = unidecode.unidecode(speaker)
    speaker = re.sub(r'[^\w\s]', '', speaker)  # Remove special characters including hyphens
    speaker = re.sub(r'\s+', ' ', speaker).strip()  # Remove extra spaces within and around the name
    return speaker

# Function to normalize party and country by removing symbols but preserving capitalization
def normalize_party_country(value):
    if pd.isna(value):
        return value
    # Remove special characters but keep uppercase letters
    value = unidecode.unidecode(value)  # Transliterate
    value = re.sub(r'[^\w\s]', '', value)  # Remove special characters
    return value.strip()  # Remove trailing spaces

# Function to count matching word chains, require all chains to match for a valid match
def count_matching_chains(source_name, target_name):
    source_chains = set(tokenize_name(source_name))  # Use set to allow set operations
    target_chains = set(tokenize_name(target_name))  # Use set to allow set operations

    # Only consider a match if all target chains are in source chains, or vice versa
    if target_chains.issubset(source_chains) or source_chains.issubset(target_chains):
        return len(target_chains)  # Return the number of matching chains (total chains)
    
    return 0  # No match if all chains do not match

# Paths to the source and target CSV files
source_path = 
target_path = 

# Try reading the source and target CSVs with multiple encodings
source_df, source_encoding = read_csv_with_multiple_encodings(source_path)
target_df, target_encoding = read_csv_with_multiple_encodings(target_path)

print(f"Source file loaded with encoding: {source_encoding}")
print(f"Target file loaded with encoding: {target_encoding}")

# Normalize the 'Country' and 'Party' columns in target_df before any further processing
target_df['Country'] = target_df['Country'].apply(lambda x: normalize_party_country(x))
target_df['Party'] = target_df['Party'].apply(lambda x: normalize_party_country(x))

# Clean the 'Speaker' column in target_df
target_df['Speaker'] = target_df['Speaker'].apply(lambda x: clean_speaker_column(x))

# Check for rows with empty Party and Country, excluding those where the speaker is "President" in any language
president_titles = ["president", "président", "präsident"]  # Add any other language variations

# Modify the lambda function to handle NaN values in the 'Speaker' column and ignore them
empty_rows = target_df[
    target_df['Speaker'].apply(lambda x: isinstance(x, str) and x.lower() not in president_titles) & 
    (target_df['Party'].isna() | target_df['Country'].isna())
]

# Print the number of rows with missing 'Party' and 'Country' data
print(f"Rows with missing Party or Country data (excluding President rows): {len(empty_rows)} out of {len(target_df)} total rows")

# Initialize counters for matches found, not found, and changes made
matches_found = 0
matches_not_found = 0
changes_made = 0  # Counter for changes made to Party or Country

# Iterate only through rows with missing Party or Country data
for target_index, target_row in empty_rows.iterrows():
    # Skip rows where the speaker is "President" in any language
    if target_row['Speaker'].lower() in president_titles:
        continue

    max_matches = 0
    best_match_source_row = None

    for source_index, source_row in source_df.iterrows():
        matching_chains = count_matching_chains(source_row['Name'], target_row['Speaker'])
        if matching_chains > max_matches:
            max_matches = matching_chains
            best_match_source_row = source_row

    # Print the result for this target row
    if best_match_source_row is not None:
        print(f"Target Speaker '{target_row['Speaker']}' matches {max_matches} chains with source name '{best_match_source_row['Name']}'")
    else:
        print(f"No match found for Target Speaker '{target_row['Speaker']}'")

    # If a match is found with a significant number of chains, fill in missing data
    if max_matches > 0 and best_match_source_row is not None:
        if pd.isna(target_row['Country']):
            target_df.at[target_index, 'Country'] = best_match_source_row['Country']
            changes_made += 1
        if pd.isna(target_row['Party']):
            target_df.at[target_index, 'Party'] = best_match_source_row['Party']
            changes_made += 1
        matches_found += 1
    else:
        matches_not_found += 1

# Create output directory if it does not exist
output_dir = r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\preprocessing\2009-2014_CRE-7\2014\2014_crossed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the updated target DataFrame to a new CSV file
output_path = os.path.join(output_dir, "2014_crosseds.csv")
target_df.to_csv(output_path, index=False)

# Print the number of matches found, not found, and changes made
print(f"Total matches found: {matches_found}")
print(f"Total matches not found: {matches_not_found}")
print(f"Total changes made (Party/Country updated): {changes_made}")

# At the very end, reformat the 'Speaker' column to ensure readability (separate words)
def reformat_speaker_column(speaker):
    if pd.isna(speaker):
        return speaker
    speaker = re.sub(r'[^\w\s]', '', speaker)  # Ensure special characters are removed
    speaker = re.sub(r'\s+', ' ', speaker).strip()  # Remove extra spaces and ensure readability
    return speaker  # Return the cleaned and formatted name

# Apply the reformatting function to ensure names are properly spaced at the end
target_df['Speaker'] = target_df['Speaker'].apply(lambda x: reformat_speaker_column(x))

# Print a sample of the updated DataFrame to verify the final speaker format
print("\nUpdated DataFrame (sample with reformatted speaker names):")
print(target_df.head())



In [ ]:
#Greek and bulgarian text crossing
import os
import pandas as pd
import re
import unidecode

# Function to fix over-encoded (double-encoded) text
def fix_over_encoded_text(text):
    if isinstance(text, str):
        try:
            # First encode as ISO-8859-1 bytes, then decode back as UTF-8
            return text.encode('latin1').decode('utf-8')
        except (UnicodeDecodeError, AttributeError):
            return text
    return text

# Function to convert ISO-8859-1 to UTF-8
def convert_iso_to_utf8(file_path):
    # Open the file with ISO-8859-1 encoding and convert to UTF-8
    with open(file_path, 'r', encoding='ISO-8859-1') as f:
        content = f.read()
    
    # Write the content back in UTF-8
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"Converted {file_path} from ISO-8859-1 to UTF-8")

# Function to try reading the CSV file with different encodings
def read_csv_with_multiple_encodings(file_path):
    encodings = ['utf-8', 'ISO-8859-1', 'windows-1252', 'latin1']
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding), encoding
        except UnicodeDecodeError as e:
            print(f"Failed to read {file_path} with encoding {encoding}: {e}")
    raise UnicodeDecodeError(f"Could not decode {file_path} with the tried encodings.")

# Function to detect if the quote contains Greek or Bulgarian characters
def contains_greek_or_bulgarian(text):
    if pd.isna(text):
        return False
    # Greek Unicode range: U+0370 to U+03FF, Bulgarian Cyrillic range: U+0400 to U+04FF
    greek_bulgarian_pattern = re.compile(r'[\u0370-\u03FF\u0400-\u04FF]')
    return bool(greek_bulgarian_pattern.search(text))

# Paths to the source and target CSV files
source_path = 
target_path = 
# Convert both source and target files from ISO-8859-1 to UTF-8
convert_iso_to_utf8(source_path)
convert_iso_to_utf8(target_path)

# Try reading the source and target CSVs with multiple encodings
source_df, source_encoding = read_csv_with_multiple_encodings(source_path)
target_df, target_encoding = read_csv_with_multiple_encodings(target_path)

print(f"Source file loaded with encoding: {source_encoding}")
print(f"Target file loaded with encoding: {target_encoding}")

# Fix over-encoded text in the 'Quote' column first
target_df['Quote'] = target_df['Quote'].apply(fix_over_encoded_text)

# Normalize the 'Country' and 'Party' columns in target_df before any further processing
target_df['Country'] = target_df['Country'].apply(lambda x: unidecode.unidecode(x) if pd.notnull(x) else x)
target_df['Party'] = target_df['Party'].apply(lambda x: unidecode.unidecode(x) if pd.notnull(x) else x)

# Clean the 'Speaker' column in target_df
target_df['Speaker'] = target_df['Speaker'].apply(lambda x: unidecode.unidecode(x) if pd.notnull(x) else x)

# Check for rows with empty Party and Country, excluding those where the speaker is "President" in any language
president_titles = ["president", "président", "präsident"]

# Only consider rows with Greek or Bulgarian quotes and missing Party/Country
empty_rows = target_df[
    target_df['Speaker'].apply(lambda x: isinstance(x, str) and x.lower() not in president_titles) & 
    (target_df['Party'].isna() | target_df['Country'].isna()) &
    target_df['Quote'].apply(contains_greek_or_bulgarian)
]

# Print the number of rows with missing 'Party' and 'Country' data and Greek/Bulgarian quotes
print(f"Rows with missing Party or Country data and Greek/Bulgarian quotes: {len(empty_rows)} out of {len(target_df)} total rows")

# Initialize counters for matches found, not found, and changes made
matches_found = 0
matches_not_found = 0
changes_made = 0

# Matching logic (similar to earlier logic) would go here...

# Save the updated target DataFrame to the original file (editing the target file directly)
target_df.to_csv(target_path, index=False)

# Print the number of matches found, not found, and changes made
print(f"Total matches found: {matches_found}")
print(f"Total matches not found: {matches_not_found}")
print(f"Total changes made (Party/Country updated): {changes_made}")

# Print a sample of the updated DataFrame to verify the final speaker format
print("\nUpdated DataFrame (sample):")
print(target_df.head())


